# Phase Transformation Foundations

PyTex now has the first stable primitive family for phase-transformation work:

- `OrientationRelationship`
- `TransformationVariant`
- `PhaseTransformationRecord`

This notebook explains the semantics rather than pretending the repo already has a
full parent-reconstruction platform.


In [1]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [2]:
crystal, specimen, map_frame, detector, lab, parent_phase = make_context()
child_crystal = ReferenceFrame(
    "child_crystal",
    FrameDomain.CRYSTAL,
    ("a", "b", "c"),
    Handedness.RIGHT,
)
child_symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=child_crystal)
child_lattice = Lattice(3.2, 3.2, 3.2, 90.0, 90.0, 90.0, crystal_frame=child_crystal)
child_phase = Phase(
    "child_phase",
    lattice=child_lattice,
    symmetry=child_symmetry,
    crystal_frame=child_crystal,
)

relationship = OrientationRelationship(
    name="demo_or",
    parent_phase=parent_phase,
    child_phase=child_phase,
    parent_to_child_rotation=Rotation.from_axis_angle([0.0, 0.0, 1.0], np.pi / 4.0),
)

variants = relationship.generate_variants()
print("Variant count:", len(variants))


Variant count: 3


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [3]:
parent_orientation = Orientation(
    rotation=Rotation.identity(),
    crystal_frame=parent_phase.crystal_frame,
    specimen_frame=specimen,
    symmetry=parent_phase.symmetry,
    phase=parent_phase,
)
child_orientations = OrientationSet.from_orientations(
    [
        Orientation(
            rotation=relationship.parent_to_child_rotation,
            crystal_frame=child_phase.crystal_frame,
            specimen_frame=specimen,
            symmetry=child_phase.symmetry,
            phase=child_phase,
        )
    ]
)
record = PhaseTransformationRecord(
    name="demo_record",
    orientation_relationship=relationship,
    parent_orientation=parent_orientation,
    child_orientations=child_orientations,
    variant_indices=np.array([1]),
)
print(record.variant_count)


1
